# 05. UGV 주행 및 위치 확인 (Driving And Odometry)

영상 대응: **24:35–25:17**

> 3차 교육 영상 20–30분의 기능을 재구성한 **로컬 실습본**입니다. 공식 원본 노트북이 아닙니다. `mil_msgs` 원본 정의가 없어 이 환경에서는 `sim_mil_interfaces`를 사용합니다. 실제 ROS 2로 시뮬레이터와 통신합니다.

`/cmd_vel`에 `geometry_msgs/msg/Twist`를 발행하고 `/odom`의 `nav_msgs/msg/Odometry`를 구독합니다.

| 항목 | 값 |
|---|---|
| 주행 명령 주기 | 10Hz, 초당 10회 |
| 위치 갱신 주기 | 교육자료 기준 20Hz |
| linear.x | 전진(+), 후진(-), -0.6~0.6 m/s |
| angular.z | 좌회전(+), 우회전(-), -3~3 rad/s |
| position.x / y | 출발점 기준 전후 / 좌우 위치(m) |
| orientation | 쿼터니언 x, y, z, w. yaw로 변환 가능 |

영상처럼 `RUN_DRIVE=False`가 기본입니다. 먼저 위치만 읽어 보고, 값을 True로 바꾸면 정해진 시간 동안 주행합니다. 센서가 늦거나 경기 제어가 잠기면 정지하며, 정상 종료·중단·오류에서도 0속도 명령을 보냅니다.


## 실행 방법

위쪽 커널을 **Python 3 (ROS 2 · simulation)** 으로 선택하고, 아래 셀을 위에서부터 **Shift+Enter**로 실행합니다. 물리 경기장과 ROS 실습 연결을 먼저 켜세요. 각 실습은 시간이 지나면 종료되며, 셀을 다시 실행할 수 있습니다.


In [ ]:
import time
import math
import uuid
import rclpy
from rclpy.qos import QoSProfile, ReliabilityPolicy, DurabilityPolicy, qos_profile_sensor_data
from rclpy.signals import SignalHandlerOptions
from sim_mil_interfaces.msg import State as MissionState, Match as MatchCommand

# 상태/경기 토픽은 마지막 값을 새 구독자에게도 전달합니다.
STATUS_QOS = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                        durability=DurabilityPolicy.TRANSIENT_LOCAL)

def new_node(prefix):
    if not rclpy.ok():
        # Jupyter의 중단 버튼이 KeyboardInterrupt와 finally 정지를 처리하게 둡니다.
        rclpy.init(args=[], signal_handler_options=SignalHandlerOptions.NO)
    return rclpy.create_node(prefix + '_' + uuid.uuid4().hex[:8])

def spin_until(node, predicate, timeout=5.0):
    deadline = time.monotonic() + timeout
    while not predicate() and time.monotonic() < deadline:
        rclpy.spin_once(node, timeout_sec=min(0.05, max(0.0, deadline-time.monotonic())))
    if not predicate():
        raise TimeoutError('ROS 데이터/응답 시간 초과. 물리 경기장과 ROS 연결 상태를 확인하세요.')

def watch_status(node):
    data = {'state': None, 'match': None, 'state_at': None}
    def on_state(msg):
        data['state'] = msg
        data['state_at'] = time.monotonic()
    def on_match(msg):
        data['match'] = msg
    subscriptions = [
        node.create_subscription(MissionState, '/mil/state', on_state, STATUS_QOS),
        node.create_subscription(MatchCommand, '/mil/match', on_match, STATUS_QOS),
    ]
    spin_until(node, lambda: data['state'] is not None and data['match'] is not None)
    return data, subscriptions

def allowed(data):
    if data['state_at'] is None or time.monotonic()-data['state_at'] > 1.5:
        return False
    match = data['match']
    if match is None or match.cmd == 'pause':
        return False
    return match.mode == 'practice' or (match.mode == 'mission' and match.cmd in ('start', 'resume'))

def require_control(data):
    if not allowed(data):
        raise RuntimeError('제어 잠금 또는 상태 데이터 지연. 경기 재개·연결 상태를 확인하세요.')

def wait_publisher(node, publisher, timeout=5.0):
    spin_until(node, lambda: publisher.get_subscription_count() > 0, timeout)

def yaw_degrees(q):
    return math.degrees(math.atan2(2.0*(q.w*q.z + q.x*q.y), 1.0-2.0*(q.y*q.y+q.z*q.z)))

def source_age_seconds(node, message, maximum, label):
    stamp = message.header.stamp
    source_ns = int(stamp.sec)*1_000_000_000 + int(stamp.nanosec)
    if source_ns <= 0:
        raise ValueError(label + ': 원본 센서 시각이 없습니다.')
    age = (node.get_clock().now().nanoseconds-source_ns)/1_000_000_000
    if age < -0.25:
        raise ValueError(f'{label}: 센서 시각이 {-age:.3f}초 미래입니다. ROS 시계 설정을 확인하세요.')
    if age > maximum:
        raise TimeoutError(f'{label}: 원본 데이터가 {age:.3f}초 전 값입니다.')
    return age


## 실습 설정

아래 값을 바꾸고 다시 실행할 수 있습니다.


In [ ]:
RUN_DRIVE = False
LINEAR_X = 0.10
ANGULAR_Z = 0.0
DURATION_SECONDS = 2.0
STALE_SECONDS = 0.75


## 실행


In [ ]:
from geometry_msgs.msg import Twist
from nav_msgs.msg import Odometry
if not (-0.6 <= LINEAR_X <= 0.6 and -3.0 <= ANGULAR_Z <= 3.0):
    raise ValueError('주행 속도 범위를 확인하세요.')
if not 0.0 < float(DURATION_SECONDS) <= 10.0:
    raise ValueError('이 실습의 주행 시간은 0초 초과~10초입니다.')
node = new_node('demo_driving')
publisher = None
latest = {'odom':None, 'at':None}
def on_odom(msg):
    latest['odom'], latest['at'] = msg, time.monotonic()
try:
    subscription = node.create_subscription(Odometry, '/odom', on_odom, qos_profile_sensor_data)
    data, subscriptions = watch_status(node)
    spin_until(node, lambda: latest['odom'] is not None)
    odom_source_age = source_age_seconds(node, latest['odom'], STALE_SECONDS, 'Odometry')
    before = latest['odom'].pose.pose.position
    start_xy = [float(before.x), float(before.y)]
    if RUN_DRIVE:
        publisher = node.create_publisher(Twist, '/cmd_vel', 10)
        wait_publisher(node, publisher)
        end = time.monotonic() + float(DURATION_SECONDS)
        next_send = 0.0
        while time.monotonic() < end:
            rclpy.spin_once(node, timeout_sec=0.02)
            require_control(data)
            if time.monotonic()-latest['at'] > STALE_SECONDS:
                raise TimeoutError('위치 데이터 지연: 주행을 멈춥니다.')
            odom_source_age = source_age_seconds(node, latest['odom'], STALE_SECONDS, 'Odometry')
            if time.monotonic() >= next_send:
                msg = Twist()
                msg.linear.x, msg.angular.z = float(LINEAR_X), float(ANGULAR_Z)
                publisher.publish(msg)
                next_send = time.monotonic() + 0.1
        publisher.publish(Twist())
        until = time.monotonic() + 0.25
        while time.monotonic() < until:
            rclpy.spin_once(node, timeout_sec=0.02)
    odom_source_age = source_age_seconds(node, latest['odom'], STALE_SECONDS, 'Odometry')
    position = latest['odom'].pose.pose
    yaw = yaw_degrees(position.orientation)
    print(f'Position: X={position.position.x:.3f}m, Y={position.position.y:.3f}m | Yaw={yaw:.1f}deg')
    print('주행 실행' if RUN_DRIVE else '위치 확인만 실행: RUN_DRIVE = False')
    DEMO_RESULT = {'demo':'05', 'drive_enabled':bool(RUN_DRIVE), 'start_xy':start_xy,
                   'end_xy':[float(position.position.x),float(position.position.y)], 'yaw_deg':yaw,
                   'source_age_seconds':odom_source_age,'received_age_seconds':time.monotonic()-latest['at']}
finally:
    if publisher is not None:
        for _ in range(3):
            publisher.publish(Twist())
            time.sleep(0.03)
        print('Stop command sent')
    node.destroy_node()


[← 실습 목차](00.실습목차.ipynb) · [코드 편집](../team_node.py)

실물 장치의 토픽 타입·주기·응답과 맞는지는 지급 UGV에서 확인해야 합니다.
